# Semana 4-6 - Visualización e integración

## Visualización con Seaborn y conversión desde Polars
En este cuaderno usamos Polars para preparar los datos y Seaborn/Matplotlib para graficar.
Cada celda de código va acompañada de una breve explicación con propósito, uso y parámetros habituales.


### Preparación de librerías y datos base
- Propósito: importar dependencias y cargar un dataset conocido (`tips`).
- Uso: `sns.load_dataset("tips")` devuelve un `DataFrame` pandas; lo convertimos a Polars con `pl.from_pandas`.
- Parámetros clave: `load_dataset(name, cache=True)` (usa caché por defecto); `pl.Config.set_fmt_str_lengths` solo ajusta impresión.


In [ ]:
import pandas as pd
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

sns.set_theme(style="whitegrid")

pdf_tips = sns.load_dataset("tips")
pldf_tips = pl.from_pandas(pdf_tips)

pl.Config.set_fmt_str_lengths(12)
pdf_tips.head(), pldf_tips.head()

### 1. Conversión de Polars a pandas para graficar
- Propósito: muchos gráficos siguen esperando pandas o arrays; convertimos con `.to_pandas()`.
- Uso: `pldf.to_pandas()` crea una copia en pandas; evita modificar el original.
- Parámetros: sin argumentos; admite `use_pyarrow=False` por defecto (convierte a tipos nativos).


In [ ]:
pdf_for_plot = pldf_tips.to_pandas()
pdf_for_plot.info()

### 2. Histograma y KDE de la propina
- Propósito: ver la distribución de `tip` rápidamente.
- Uso: `sns.histplot(data=df, x="col", kde=True, bins=20)`; `kde` activa densidad, `bins` 20 por defecto 10.
- Parámetros comunes: `hue=None`, `stat="count"` por defecto.


In [ ]:
pdf = pldf_tips.to_pandas()
ax = sns.histplot(data=pdf, x="tip", kde=True, bins=20, color="#4C78A8")
ax.set_title("Distribución de propina")
plt.tight_layout()

### 3. Gráfico de barras por día con agregación en Polars
- Propósito: resumir importes medios por día usando Polars y graficar en pandas/Seaborn.
- Uso: `df.group_by("day").agg(pl.mean("total_bill"))` y luego `.to_pandas()`.
- Parámetros: `group_by` acepta lista de columnas; `pl.mean` usa `null_strategy="ignore"` por defecto.


In [ ]:
pl_day = pldf_tips.group_by("day").agg([
    pl.mean("total_bill").alias("total_bill_mean"),
    pl.mean("tip").alias("tip_mean"),
])
ax = sns.barplot(data=pl_day.to_pandas(), x="day", y="total_bill_mean", color="#72B7B2")
ax.set_title("Cuenta media por día")
ax.set_ylabel("Media total_bill")
plt.tight_layout()

### 4. Dispersión con color por turno
- Propósito: explorar relación entre `total_bill` y `tip` coloreando por `time` (almuerzo/cena).
- Uso: `sns.scatterplot(data=df, x=..., y=..., hue="time", style="sex")`.
- Parámetros: `alpha=0.8` controla transparencia (1 por defecto); `s` tamaño de puntos (40 por defecto).


In [ ]:
pdf = pldf_tips.to_pandas()
ax = sns.scatterplot(data=pdf, x="total_bill", y="tip", hue="time", style="sex", alpha=0.8)
ax.set_title("Propina vs cuenta total")
plt.tight_layout()

### 5. Boxplot y Violin plot por día
- Propósito: comparar distribución de propina por día con dos vistas complementarias.
- Uso: `sns.boxplot(data=df, x="day", y="tip")` y `sns.violinplot(...)`.
- Parámetros: `showfliers=True` en boxplot por defecto; `inner="box"` en violin muestra caja interna.


In [ ]:
pdf = pldf_tips.to_pandas()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(data=pdf, x="day", y="tip", ax=axes[0], color="#E45756")
axes[0].set_title("Boxplot propina por día")

sns.violinplot(data=pdf, x="day", y="tip", ax=axes[1], inner="box", color="#F58518")
axes[1].set_title("Violin propina por día")
plt.tight_layout()

### 6. Densidad 2D (hexbin) con Matplotlib
- Propósito: ver concentración de puntos en 2D.
- Uso: `plt.hexbin(x, y, gridsize=25, cmap="viridis")`.
- Parámetros: `gridsize=25` (por defecto 100); `mincnt=None` (permite celdas vacías).


In [ ]:
pdf = pldf_tips.to_pandas()
plt.figure(figsize=(5,4))
plt.hexbin(pdf["total_bill"], pdf["tip"], gridsize=25, cmap="viridis")
plt.colorbar(label="Cuenta de puntos")
plt.title("Hexbin total vs tip")
plt.xlabel("total_bill")
plt.ylabel("tip")
plt.tight_layout()

### 7. Facetas por día y turno
- Propósito: comparar distribuciones subdividiendo por categorías.
- Uso: `sns.displot(..., col="day", row="time", kind="hist")` crea un grid de subplots.
- Parámetros: `col_wrap=None` por defecto; `binwidth=None` deja que seaborn infiera bins.


In [ ]:
pdf = pldf_tips.to_pandas()
g = sns.displot(data=pdf, x="tip", col="day", row="time", kind="hist", bins=15, color="#54A24B")
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle("Propina por día y turno")
plt.tight_layout()

### 8. Heatmap de correlación
- Propósito: ver correlación entre variables numéricas preparadas en Polars.
- Uso: Polars calcula la matriz con `df.select(pl.corr(col1, col2))` o convirtiendo a pandas y usando `.corr()`.
- Parámetros: `sns.heatmap(data, annot=True, cmap="coolwarm")`; `annot` por defecto False.


In [ ]:
num_cols = ["total_bill", "tip", "size"]
pdf_corr = pldf_tips.select(num_cols).to_pandas().corr()
ax = sns.heatmap(pdf_corr, annot=True, cmap="coolwarm", fmt=".2f")
ax.set_title("Correlación numérica")
plt.tight_layout()

### 9. Gráfico de líneas con agregación temporal simulada
- Propósito: mostrar evolución diaria generando fechas con Polars.
- Uso: creamos una columna de fecha sintética y agregamos: `with_columns(pl.date_range(...))` y `group_by("fecha").sum`.
- Parámetros: `pl.date_range(low, high, interval="1d")`; `interval` por defecto 1 día si se pasa un `timedelta`.


In [ ]:
pl_temporal = (
    pldf_tips
    .with_row_index(name="idx")
    .with_columns((pl.date(2024, 1, 1) + pl.col("idx").cast(pl.Int64)).alias("fecha"))
)
pl_daily = pl_temporal.group_by("fecha").agg(pl.sum("tip").alias("tip_total"))
ax = sns.lineplot(data=pl_daily.to_pandas(), x="fecha", y="tip_total", marker="o", color="#B279A2")
ax.set_title("Propina total por día (simulado)")
plt.xticks(rotation=45)
plt.tight_layout()

### 10. Catplot de conteo por fumador y turno
- Propósito: contar observaciones en categorías combinadas.
- Uso: `sns.catplot(data=df, x="smoker", hue="time", kind="count")`.
- Parámetros: `kind="count"` cuenta filas; `palette` personaliza colores (por defecto según tema).


In [ ]:
pdf = pldf_tips.to_pandas()
g = sns.catplot(data=pdf, x="smoker", hue="time", kind="count", palette="Set2")
g.fig.suptitle("Recuento por hábito de fumar y turno")
plt.tight_layout()

### 11. Pairplot para relaciones múltiples
- Propósito: explorar pares de variables numéricas con color por sexo.
- Uso: `sns.pairplot(data=df, vars=[...], hue="sex", diag_kind="kde")`.
- Parámetros: `diag_kind="auto"` por defecto; `corner=False` muestra triángulo completo.


In [ ]:
pdf = pldf_tips.to_pandas()
g = sns.pairplot(data=pdf, vars=["total_bill", "tip", "size"], hue="sex", diag_kind="kde")
g.fig.suptitle("Pairplot numérico", y=1.02)
plt.tight_layout()

### 12. Barras apiladas con preparación en Polars
- Propósito: comparar proporciones de fumadores por día.
- Uso: pivot en Polars `pivot(values="size", index="day", columns="smoker", aggregate_function="count")` y normalizar.
- Parámetros: `aggregate_function` requiere nombre de función; `fill_value` no disponible en Polars pivot (usar `fill_null`).


In [ ]:
pl_stack = pldf_tips.pivot(values="size", index="day", columns="smoker", aggregate_function="count").fill_null(0)
pl_stack = pl_stack.with_columns([
    (pl.col("No") / (pl.col("No") + pl.col("Yes"))).alias("No_prop"),
    (pl.col("Yes") / (pl.col("No") + pl.col("Yes"))).alias("Yes_prop"),
])
plot_df = pl_stack.select(["day", "No_prop", "Yes_prop"]).to_pandas().set_index("day")
plot_df[["No_prop", "Yes_prop"]].plot(kind="bar", stacked=True, color=["#4E79A7", "#E15759"], title="Proporción fumadores por día")
plt.tight_layout()

### 13. Guardar figuras
- Propósito: persistir un gráfico generado desde datos Polars.
- Uso: `fig.savefig(path, dpi=300, bbox_inches="tight")`.
- Parámetros: `dpi=100` por defecto; `bbox_inches='tight'` ajusta márgenes.


In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
sns.barplot(data=pl_day.to_pandas(), x="day", y="tip_mean", ax=ax, color="#59A14F")
ax.set_title("Tip media por día")
fig.savefig("plot_tip_media.png", dpi=150, bbox_inches="tight")
Path("plot_tip_media.png").exists()

Limpieza opcional de archivos temporales de salida (gráficos guardados).

In [ ]:
from pathlib import Path
for path in [Path("plot_tip_media.png")]:
    if path.exists():
        path.unlink()
Path("plot_tip_media.png").exists()